# Right Augmented Chow Polynomials for Projective Geometry Matroids & Deletions

This notebook computes the Right Augmented Chow polynomials ($F = \underline{H} \cdot f^{\text{rev}}$) for various projective geometry matroids $PG(r-1, n)$ and their single-element deletions $M \setminus e$, verifying their properties such as symmetry and equality ($F(M) = F(M \setminus e)$ for $r \ge 3$).

In [1]:
import sys
import os
# Get current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all custom matroid functions
from custom_matroid_functions import *

In [2]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()

# Kazhdan-Lusztig polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)

# Reversal map: f^{rev}(M) = x^{rank(M)} * f(M)(x^{-1})
rev = lambda f: lambda M: f(M)(x**-1) * x**M.rank()

# Convolution: (f * g)(M) = sum_{F in L(M)} f(M|_F) * g(M/F)
def convolve(f, g):
    return lambda M: sum([
        f(M.delete(M.groundset() - F))(x) * g(M.contract(F))(x) 
        for F in list(M.lattice_of_flats())
    ])

# Right Augmented Chow polynomial
rightAugpoly = convolve(chow_polynomial, rev(kl_poly))

def check_symmetry(p, rank):
    if p == 0: return True
    return p == p(x**-1) * x**rank

In [3]:
def evaluate_pg_matroid(r, n):
    """
    Computes and prints polynomials for PG(r-1, n) and its single-element deletion M\\e.
    """
    print(f"=== Projective Geometry Matroid PG({r-1}, {n}) ===")
    try:
        M = matroids.PG(r - 1, n)
    except Exception as err:
        print(f"Failed to construct PG({r-1}, {n}): {err}\n")
        return
        
    print(f"Rank: {M.rank()}, Size: {M.size()}")
    
    # Characteristic polynomial
    cp = characteristic_polynomial(M)(x)
    print(f"Characteristic polynomial: {cp}")
    
    # Kazhdan-Lusztig polynomial
    kl = kl_poly(M)
    print(f"Kazhdan-Lusztig polynomial: {kl}")
    
    # Chow polynomial
    ch = chow_polynomial(M)
    print(f"Chow polynomial: {ch}")
    
    # Right Augmented Chow polynomial of M
    rac_M = rightAugpoly(M)
    print(f"F(M)   = {rac_M}")
    print(f"F(M) Symmetric: {check_symmetry(rac_M, M.rank())}")
    
    # Right Augmented Chow polynomial of M\\e
    e = list(M.groundset())[0]
    M_dele = M.delete(e)
    rac_dele = rightAugpoly(M_dele)
    print(f"F(M\\e) = {rac_dele}")
    print(f"F(M\\e) Symmetric: {check_symmetry(rac_dele, M_dele.rank())}")
    print(f"F(M) - F(M\\e) = {rac_M - rac_dele}")
    print("-" * 40 + "\n")

In [4]:
examples = [
    (2, 2),  # PG(1, 2) which is U_{2, 3}
    (2, 3),  # PG(1, 3) which is U_{2, 4}
    (2, 4),  # PG(1, 4) which is U_{2, 5}
    (3, 2),  # PG(2, 2) (Fano Matroid)
    (3, 3),  # PG(2, 3)
    (3, 4),  # PG(2, 4)
    (4, 2),  # PG(3, 2)
    (4, 3)   # PG(3, 3)
]

for r, n in examples:
    evaluate_pg_matroid(r, n)

=== Projective Geometry Matroid PG(1, 2) ===
Rank: 2, Size: 3
Characteristic polynomial: 2 - 3*x + x^2
Kazhdan-Lusztig polynomial: 1
Chow polynomial: x + 1
F(M)   = 1 + 4*x + x^2
F(M) Symmetric: True
F(M\e) = 1 + 3*x + x^2
F(M\e) Symmetric: True
F(M) - F(M\e) = x
----------------------------------------

=== Projective Geometry Matroid PG(1, 3) ===
Rank: 2, Size: 4
Characteristic polynomial: 3 - 4*x + x^2
Kazhdan-Lusztig polynomial: 1
Chow polynomial: x + 1
F(M)   = 1 + 5*x + x^2
F(M) Symmetric: True
F(M\e) = 1 + 4*x + x^2
F(M\e) Symmetric: True
F(M) - F(M\e) = x
----------------------------------------

=== Projective Geometry Matroid PG(1, 4) ===
Rank: 2, Size: 5
Characteristic polynomial: 4 - 5*x + x^2
Kazhdan-Lusztig polynomial: 1
Chow polynomial: x + 1
F(M)   = 1 + 6*x + x^2
F(M) Symmetric: True
F(M\e) = 1 + 5*x + x^2
F(M\e) Symmetric: True
F(M) - F(M\e) = x
----------------------------------------

=== Projective Geometry Matroid PG(2, 2) ===
Rank: 3, Size: 7
Characteristic polyn